<center><h1>Xie_Chong_HW7</h1></center>

Name: Chong Xie
<br>
Github Username: cxieusc
<br>
USC ID: 6800668300

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings(
    "ignore",
    message=r"(divide by zero|overflow|invalid value) encountered in matmul",
    category=RuntimeWarning,
    module=r"sklearn\.utils\.extmath"
)

### (a) Download the Anuran Calls (MFCCs) Data Set

In [22]:
df = pd.read_csv('../data/Anuran Calls (MFCCs)/Frogs_MFCCs.csv')
df.head(3)

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,...,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,...,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,...,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1


In [23]:
df = df.drop(columns=['RecordID'])
df_train, df_test = train_test_split(df, train_size=0.7)

In [24]:
print(f"X_train shape: {df_train.shape}")
print(f"X_test shape: {df_test.shape}")

X_train shape: (5036, 25)
X_test shape: (2159, 25)


### (b) Train a classifier for each label

#### (i) Research

**Exact Match**

Exact match measures the proportion of samples for which all three labels are predicted correctly. If any label is incorrect, the entire sample is considered incorrect. Higher values indicate better performance.

$$
\text{Exact Match}
=
\frac{1}{N}\sum_{i=1}^{N}
\mathbf{1}\left(y_{ij}=\hat{y}_{ij}\text{ for all }j\right)
$$

Reference: [scikit-learn — accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

**Hamming Loss**

Hamming loss measures the proportion of incorrectly predicted label positions. In this problem, errors are averaged over all samples and the three categorical outputs. Lower values indicate better performance.

$$
\text{Hamming Loss}
=
\frac{1}{NL}\sum_{i=1}^{N}\sum_{j=1}^{L}
\mathbf{1}(y_{ij}\neq\hat{y}_{ij})
$$


Reference: [scikit-learn — hamming_loss](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.hamming_loss.html)

**Hamming Score**

Hamming score is defined as the proportion of correctly predicted label positions. Higher values indicate better performance.

$$
\text{Hamming Score}
=
1-\text{Hamming Loss}
=
\frac{1}{NL}\sum_{i=1}^{N}\sum_{j=1}^{L}
\mathbf{1}(y_{ij}=\hat{y}_{ij})
$$

**We will use all the methods above to evaluate all the models**

#### (ii) Train a SVM for each of the labels

In [25]:
label_cols = ['Family', 'Genus', 'Species']

X_train = df_train.drop(columns=label_cols)
y_train = df_train[label_cols]
X_test = df_test.drop(columns=label_cols)
y_test = df_test[label_cols]

In [26]:
# Initial search ranges for C and gamma
param_grid = {
    "estimator__C": [0.1, 1, 10, 100],
    "estimator__gamma": [0.01, 0.1, 1, 10]
}

rbf_models = {}
rbf_results = []

y_pred_rbf = pd.DataFrame(index=y_test.index)

for label in label_cols:
    print(f"Training classifier for {label}...")

    model = OneVsRestClassifier(
        SVC(kernel="rbf")
    )

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=10,
    )

    search.fit(X_train, y_train[label])

    rbf_models[label] = search.best_estimator_
    y_pred_rbf[label] = search.predict(X_test)

    rbf_results.append({
        "Label": label,
        "Best C": search.best_params_["estimator__C"],
        "Best gamma": search.best_params_["estimator__gamma"],
        "CV Accuracy": search.best_score_,
        "Test Accuracy": (
            y_pred_rbf[label] == y_test[label]
        ).mean()
    })

rbf_results = pd.DataFrame(rbf_results)
display(rbf_results)

Training classifier for Family...
Training classifier for Genus...
Training classifier for Species...


,Label,Best C,Best gamma,CV Accuracy,Test Accuracy
0,Family,100,1,0.992453,0.991663
1,Genus,10,1,0.990270,0.990736
2,Species,10,1,0.990269,0.989810


In [27]:
correct = (
    y_test[label_cols].to_numpy()
    == y_pred_rbf[label_cols].to_numpy()
)

rbf_metrics = pd.DataFrame([{
    "Model": "RBF SVM",
    "Exact Match": correct.all(axis=1).mean(),
    "Hamming Score": correct.mean(),
    "Hamming Loss": (~correct).mean()
}])

display(rbf_metrics)

,Model,Exact Match,Hamming Score,Hamming Loss
0,RBF SVM,0.986568,0.990736,0.009264


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [31]:
param_grid_l1 = {
    "svm__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

l1_models = {}
l1_results = []
y_pred_l1 = pd.DataFrame(index=y_test.index)

for label in label_cols:
    print(f"Training L1 SVM for {label}...")

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(
            penalty="l1",
            loss="squared_hinge",
            dual=False,
            max_iter=50000
        ))
    ])

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid_l1,
        scoring="accuracy",
        cv=10,
        n_jobs=-1
    )

    search.fit(X_train, y_train[label])

    l1_models[label] = search.best_estimator_
    y_pred_l1[label] = search.predict(X_test)

    l1_results.append({
        "Label": label,
        "Best C": search.best_params_["svm__C"],
        "CV Accuracy": search.best_score_,
        "Test Accuracy": (
            y_pred_l1[label] == y_test[label]
        ).mean()
    })

l1_results = pd.DataFrame(l1_results)
display(l1_results)

Training L1 SVM for Family...


Training L1 SVM for Genus...
Training L1 SVM for Species...


,Label,Best C,CV Accuracy,Test Accuracy
0,Family,10,0.933681,0.940250
1,Genus,100,0.947383,0.951366
2,Species,100,0.957707,0.958777


In [32]:
correct_l1 = (
    y_test[label_cols].to_numpy()
    == y_pred_l1[label_cols].to_numpy()
)

l1_metrics = pd.DataFrame([{
    "Model": "L1 Linear SVM",
    "Exact Match": correct_l1.all(axis=1).mean(),
    "Hamming Score": correct_l1.mean(),
    "Hamming Loss": (~correct_l1).mean()
}])

display(l1_metrics)

,Model,Exact Match,Hamming Score,Hamming Loss
0,L1 Linear SVM,0.920333,0.950131,0.049869


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [36]:
param_grid_smote = {
    "svm__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

smote_models = {}
smote_results = []
y_pred_smote = pd.DataFrame(index=y_test.index)

for label in label_cols:
    print(f"Training L1 SVM with SMOTE for {label}...")

    model = ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE()),
        ("svm", LinearSVC(
            penalty="l1",
            loss="squared_hinge",
            dual=False,
            max_iter=50000
        ))
    ])

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid_smote,
        scoring="accuracy",
        cv=10,
        n_jobs=-1
    )

    search.fit(X_train, y_train[label])

    smote_models[label] = search.best_estimator_
    y_pred_smote[label] = search.predict(X_test)

    smote_results.append({
        "Label": label,
        "Best C": search.best_params_["svm__C"],
        "CV Accuracy": search.best_score_,
        "Test Accuracy": (
            y_pred_smote[label] == y_test[label]
        ).mean()
    })

smote_results = pd.DataFrame(smote_results)
display(smote_results)

Training L1 SVM with SMOTE for Family...
Training L1 SVM with SMOTE for Genus...
Training L1 SVM with SMOTE for Species...


/opt/homebrew/Caskroom/miniconda/base/envs/ds552/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Label,Best C,CV Accuracy,Test Accuracy
0,Family,100,0.917988,0.918018
1,Genus,10,0.917199,0.902270
2,Species,10,0.954532,0.951830


In [37]:
correct_smote = (
    y_test[label_cols].to_numpy()
    == y_pred_smote[label_cols].to_numpy()
)

smote_metrics = pd.DataFrame([{
    "Model": "L1 Linear SVM + SMOTE",
    "Exact Match": correct_smote.all(axis=1).mean(),
    "Hamming Score": correct_smote.mean(),
    "Hamming Loss": (~correct_smote).mean()
}])

comparison = pd.concat(
    [rbf_metrics, l1_metrics, smote_metrics],
    ignore_index=True
)

display(comparison)

,Model,Exact Match,Hamming Score,Hamming Loss
0,RBF SVM,0.986568,0.990736,0.009264
1,L1 Linear SVM,0.920333,0.950131,0.049869
2,L1 Linear SVM + SMOTE,0.854562,0.924039,0.075961


The RBF SVM achieved the best test performance, with an exact match
of 0.986568 and a Hamming loss of 0.009264.

Adding SMOTE to the L1-penalized linear SVM reduced exact match from
0.920333 to 0.854562 and increased Hamming loss from 0.049869 to
0.075961. Therefore, SMOTE did not improve overall performance under
these metrics in this experiment.

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

### (b) Determine which family is the majority

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

## 3. ISLR 12.6.2